# Controller Walkthrough

Explore the implemented expert controllers through short executions. Each transition reads a symbol, writes a symbol, moves the head and updates the finite control state. The environment stores the working memory.

The catalog and PTO counter require the DEL analysis environment (`pip install -e '.[analysis]'`) and no trained weights. Animations use the same simulator. Check the launch switches before running the animation cells; an enabled viewer runs until interrupted.

The examples illustrate behavior on specific inputs. Consult `analyses/` for equivalence arguments and executable checks.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ["DEL_REPO_ROOT"]) if os.environ.get("DEL_REPO_ROOT") else next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/differentiable_exact_learning").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import display
from differentiable_exact_learning.viewer import Session, load_controller, open_viewer
from differentiable_exact_learning.analysis.plot_style import apply_paper_style
apply_paper_style()


## Controller Catalog

The catalog reads `expert.n_states`, including the halting state. These counts describe implemented controllers before minimization and do not measure neural hidden dimensions. Arithmetic controllers support bases 2–10; the table shows binary and decimal instances. Alternative grid controllers use the `new` variant.

The universal teleporting agent has 15 states regardless of the encoded program. A direct tape program has a separate state count. The animation example uses `tp:copy`; use another `tp:<program>` selector to change the encoded program.

In [2]:
SELECTORS = ['grid:add2', 'grid:add10', 'grid:mult2', 'grid:mult10', 'grid_new:add2', 'grid_new:add10', 'grid_new:mult2', 'grid_new:mult10', 'tm:add2', 'tm:add10', 'tm:mult2', 'tm:mult10', 'tm:parity', 'tm:flip', 'tm:cum_sum', 'tm:bit_delete', 'tm:shift_left', 'tm:shift_right', 'tm:reverse', 'tm:copy', 'tp:copy']
catalog = []
for selector in SELECTORS:
    expert = load_controller(selector).expert
    catalog.append({"selector": selector, "states_including_halt": expert.n_states,
                    "halt_state": expert.halt_state})
display(pd.DataFrame(catalog))

,selector,states_including_halt,halt_state
0,grid:add2,11,10
1,grid:add10,19,18
2,grid:mult2,32,31
3,grid:mult10,108,107
4,grid_new:add2,15,14
5,grid_new:add10,23,22
6,grid_new:mult2,26,25
7,grid_new:mult10,42,41
8,tm:add2,9,8
9,tm:add10,25,24


## Measure PTO Length

PTO length counts executed expert transitions, including the transition into halt exactly once. For `tp:<program>`, this counts transitions of the universal teleporting agent; `tm:<program>` counts direct tape-program transitions. The two measure different execution costs.

Choose a controller selector and an input below. Arithmetic selectors include the radix (`grid:add10`, for example). Both arithmetic operands may have different lengths; binary programs accept a binary string, including an empty string.

The counter advances the same expert simulator used by the viewer without storing the full trace. `MAX_STEPS` bounds each execution. An error or exhausted budget raises an exception; a partial execution is never reported as a completed PTO.

In [ ]:
from time import perf_counter

def count_pto(selector, text, *, max_steps):
    session = Session(load_controller(selector), text, max_steps=max_steps)
    started = perf_counter()
    while session.status not in {"halted", "limit", "error"}:
        session.advance()
    if session.status != "halted":
        raise RuntimeError(f"{selector}, input {text!r}: {session.message}")
    return {"controller": selector, "input": text, "pto_length": session.step,
            "seconds": perf_counter() - started}

CONTROLLER = "grid:mult10"
INPUT = "99x99"
MAX_STEPS = 100_000
# Binary program example: CONTROLLER = "tp:copy"; INPUT = "1011"
display(pd.DataFrame([count_pto(CONTROLLER, INPUT, max_steps=MAX_STEPS)]))


## Animation

Set `LAUNCH_ANIMATION = True` and run the next cell to open one local viewer. Choose a selector from the catalog and a matching input. The cell serves the viewer until you interrupt the cell; interrupting closes the server. Pause playback to step through transitions or inspect rune locations. Reset returns to the initial configuration.

Use a local Python kernel for the simplest setup. With a remote kernel, `127.0.0.1` refers to the remote machine; access requires forwarding the chosen port. GitHub previews cannot launch the animation server.

In [3]:
LAUNCH_ANIMATION = True
CONTROLLER = "tp:copy"
INPUT = "1011"
if LAUNCH_ANIMATION:
    open_viewer(CONTROLLER, INPUT, width=21, height=9, max_steps=10000)


Viewer: http://127.0.0.1:37285
Controller: Universal · teleporting expert
Press Ctrl-C to close.


## Inspect a Learned Controller

Supply a complete released checkpoint to play back the neural controller. The checkpoint restores the saved model and environment settings. The example uses the released binary-addition DFST; change both the path and input for another task. For a universal checkpoint, also supply `task="copy"` (or another supported program).

Playback follows autonomous execution: each prediction changes the environment and determines the next observation. Playback illustrates behavior and mistakes on the chosen input. The results notebooks report the separate teacher-forced benchmark evaluations. Neural hidden dimensions differ from extracted-controller state counts.

In [ ]:
LAUNCH_LEARNED_ANIMATION = False
CHECKPOINT_PATH = ROOT / "runs/DFST/add2/warmup_50000_steps_500000/final_model_checkpoint.pt"
if LAUNCH_LEARNED_ANIMATION:
    if not CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(CHECKPOINT_PATH)
    open_viewer(CHECKPOINT_PATH, "101+11", device="cpu", max_steps=10000)


## Command-Line Examples and Source

```sh
del animate grid:add2 '101+11'
del animate grid:mult10 '12x34'
del animate tm:flip '1011'
del animate tp:copy '1011'
```

The [grid transition tables](../src/differentiable_exact_learning/data/transitions/grid.py), [arithmetic tape tables](../src/differentiable_exact_learning/data/transitions/tm.py), and [binary programs and universal teleporting agent](../src/differentiable_exact_learning/data/transitions/teleporting.py) define the experts. The [viewer](../src/differentiable_exact_learning/viewer/controller.py) executes expert transitions. See the [analysis index](../analyses/README.md) for extracted-controller proofs and the [recipe guide](../configs/README.md) for training.